# 🎯 Chapter 05: Riparian Buffers, Encroachment & Hospital Voronoi Zones
## Spatial Overlays along the Odaw Basin and Healthcare Catchments

### 🎯 The Mission: Why We Are Doing This
The **Odaw River Basin and Korle Lagoon** suffer from severe flood risk due to building encroachment inside statutory river protection buffers ($100	ext{m}$). In this chapter, we use `ST_Buffer` and `ST_Intersection` to identify encroaching structures, and construct **Voronoi / Dirichlet polygons** for hospital service territories.

---

### 🔑 Spatial DBA Keywords Decoded

| Function | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **`ST_Buffer(geom, dist)`** | Expands a geometry outward by metric radius $d$. | Used for statutory ecological and setback zones. |
| **`ST_Intersection(A, B)`** | Returns the exact geometric overlap portion of two layers. | Extracts the precise slice of an encroaching footprint. |
| **`ST_VoronoiPolygons(pts)`** | Mathematical tessellation where every point in a cell is closest to that seed point. | Generates baseline facility service domains. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_buffer = """
DROP TABLE IF EXISTS odaw_waterway CASCADE;
CREATE TABLE odaw_waterway (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    geom GEOMETRY(LineString, 32630)
);

INSERT INTO odaw_waterway (name, geom)
VALUES ('Odaw River Main Channel', ST_GeomFromText('LINESTRING(808000 617000, 808500 615500, 809000 613000, 809200 611000)', 32630));

DROP TABLE IF EXISTS odaw_100m_buffer CASCADE;
CREATE TABLE odaw_100m_buffer AS
SELECT 1 AS id, '100m Statutory Riparian Buffer' AS description, ST_Buffer(geom, 100.0) AS geom
FROM odaw_waterway;

DROP TABLE IF EXISTS riparian_zone_buildings CASCADE;
CREATE TABLE riparian_zone_buildings (
    bld_id SERIAL PRIMARY KEY,
    bld_type VARCHAR(50),
    geom GEOMETRY(Polygon, 32630)
);

INSERT INTO riparian_zone_buildings (bld_type, geom)
VALUES 
    ('informal_residential', ST_GeomFromText('POLYGON((808480 615500, 808510 615500, 808510 615520, 808480 615520, 808480 615500))', 32630)),
    ('commercial_store', ST_GeomFromText('POLYGON((808580 615500, 808610 615500, 808610 615520, 808580 615520, 808580 615500))', 32630)),
    ('formal_office', ST_GeomFromText('POLYGON((808700 615500, 808740 615500, 808740 615530, 808700 615530, 808700 615500))', 32630));
"""

execute_sql(sql_buffer)
print("✅ Riparian buffer and building models created.")

In [ ]:
# Identify Buffer Violations
df_violations = query_df("""
SELECT 
    b.bld_id, b.bld_type,
    ST_Area(b.geom) AS total_footprint_m2,
    ROUND(ST_Area(ST_Intersection(b.geom, buf.geom))::NUMERIC, 1) AS encroached_area_m2,
    ROUND((ST_Area(ST_Intersection(b.geom, buf.geom)) / ST_Area(b.geom) * 100.0)::NUMERIC, 1) AS pct_encroached
FROM riparian_zone_buildings b
JOIN odaw_100m_buffer buf ON ST_Intersects(b.geom, buf.geom);
""")

df_violations

In [ ]:
# Interactive Map Preview: Buffer and Encroaching Buildings
gdf_overlay = query_gdf("""
SELECT bld_id AS id, bld_type AS label, geom FROM riparian_zone_buildings
UNION ALL
SELECT 999 AS id, description AS label, geom FROM odaw_100m_buffer;
""")

preview_map(gdf_overlay, tooltip_cols=["label"])